In [ ]:
!pip -q install faiss-cpu rank-bm25


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 57.3 MB/s eta 0:00:00


In [ ]:
!pip -q install sentence-transformers faiss-cpu rank-bm25 openpyxl pandas tqdm langchain-text-splitters

In [3]:
import os
import pandas as pd
import os
import requests
import json

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install faiss-cpu


In [ ]:
!pip -U install langchain


Usage:   
  pip3 <command> [options]

no such option: -U


In [ ]:
"""
RAG Source-Aware - VERSIONE CORRETTA
=====================================
FIX:
1. Skippa file .txt vuoti durante indicizzazione
2. NON usa fallback quando source non trovata → ritorna lista vuota
3. Gestisce correttamente indici vuoti
"""

import os
import json
import hashlib
import pickle
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Union, Tuple
from tqdm.auto import tqdm

from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
from rank_bm25 import BM25Okapi
from langchain_core.prompts import ChatPromptTemplate


# ============================================================================
# VECTORIZER
# ============================================================================

class Vectorizer:
    def __init__(self, model_name: str):
        from sentence_transformers import SentenceTransformer
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)

    def get_embeddings(self, texts: Union[str, List[str]]) -> np.ndarray:
        if isinstance(texts, str):
            texts = [texts]
        emb = self.model.encode(
            texts,
            normalize_embeddings=True,
            show_progress_bar=False
        )
        return np.asarray(emb, dtype=np.float32)


# ============================================================================
# CHUNK RECORD
# ============================================================================

@dataclass
class ChunkRecord:
    chunk_id: str
    source_id: str
    source_title: str
    source_path: str
    text: str


# ============================================================================
# LOCAL INDEX
# ============================================================================

class LocalIndex:
    def __init__(self, index_dir: str):
        self.index_dir = index_dir
        os.makedirs(self.index_dir, exist_ok=True)

        self.chunks_path = os.path.join(self.index_dir, "chunks.jsonl")
        self.faiss_path = os.path.join(self.index_dir, "faiss.index")
        self.meta_path = os.path.join(self.index_dir, "meta.pkl")
        self.bm25_path = os.path.join(self.index_dir, "bm25.pkl")
        self.conf_path = os.path.join(self.index_dir, "conf.json")

        self._chunks: List[ChunkRecord] = []
        self._faiss = None
        self._emb_matrix: Optional[np.ndarray] = None
        self._bm25 = None
        self._bm25_tokens: List[List[str]] = []

    def save_conf(self, conf: Dict[str, Any]) -> None:
        with open(self.conf_path, "w", encoding="utf-8") as f:
            json.dump(conf, f, ensure_ascii=False, indent=2)

    def add_chunks(self, chunks: List[ChunkRecord]) -> None:
        self._chunks.extend(chunks)

    def save_chunks(self) -> None:
        with open(self.chunks_path, "w", encoding="utf-8") as f:
            for ch in self._chunks:
                f.write(json.dumps(ch.__dict__, ensure_ascii=False) + "\n")

    def load(self) -> None:
        # Chunks
        self._chunks = []
        if os.path.exists(self.chunks_path):
            with open(self.chunks_path, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        d = json.loads(line)
                        self._chunks.append(ChunkRecord(**d))

        # FAISS
        if os.path.exists(self.faiss_path):
            self._faiss = faiss.read_index(self.faiss_path)
            with open(self.meta_path, "rb") as f:
                self._emb_matrix = pickle.load(f)

        # BM25
        if os.path.exists(self.bm25_path):
            with open(self.bm25_path, "rb") as f:
                self._bm25, self._bm25_tokens = pickle.load(f)

    def build_faiss(self, embeddings: np.ndarray) -> None:
        if len(embeddings) == 0:
            print("      ⚠️  Nessun embedding da indicizzare, skippo FAISS")
            return

        dim = embeddings.shape[1]
        self._faiss = faiss.IndexFlatIP(dim)
        self._faiss.add(embeddings)
        self._emb_matrix = embeddings

        faiss.write_index(self._faiss, self.faiss_path)
        with open(self.meta_path, "wb") as f:
            pickle.dump(self._emb_matrix, f)

    def build_bm25(self) -> None:
        if len(self._chunks) == 0:
            print("      ⚠️  Nessun chunk da indicizzare, skippo BM25")
            return

        corpus = [c.text.lower().split() for c in self._chunks]
        self._bm25 = BM25Okapi(corpus)
        self._bm25_tokens = corpus

        with open(self.bm25_path, "wb") as f:
            pickle.dump((self._bm25, self._bm25_tokens), f)

    def dense_search(self, query_emb: np.ndarray, top_k: int = 5) -> List[Dict[str, Any]]:
        if self._faiss is None or self._faiss.ntotal == 0:
            return []

        scores, indices = self._faiss.search(query_emb, min(top_k, self._faiss.ntotal))
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx >= 0 and idx < len(self._chunks):
                c = self._chunks[idx]
                results.append({
                    "chunk_id": c.chunk_id,
                    "source_id": c.source_id,
                    "source_title": c.source_title,
                    "text": c.text,
                    "score": float(score)
                })
        return results

    def sparse_search(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        if self._bm25 is None or len(self._chunks) == 0:
            return []

        tokenized_query = query.lower().split()
        scores = self._bm25.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[::-1][:top_k]

        results = []
        for idx in top_indices:
            if idx < len(self._chunks):
                c = self._chunks[idx]
                results.append({
                    "chunk_id": c.chunk_id,
                    "source_id": c.source_id,
                    "source_title": c.source_title,
                    "text": c.text,
                    "score": float(scores[idx])
                })
        return results

    def hybrid_search(self, query: str, query_emb: np.ndarray, top_k: int = 5) -> List[Dict[str, Any]]:
        dense = self.dense_search(query_emb, top_k)
        sparse = self.sparse_search(query, top_k)

        if not dense and not sparse:
            return []

        def minmax_norm(items: List[Dict[str, Any]]) -> Dict[str, float]:
            if not items:
                return {}
            vals = [it["score"] for it in items]
            mn, mx = min(vals), max(vals)
            if mx == mn:
                return {it["chunk_id"]: 1.0 for it in items}
            return {it["chunk_id"]: (it["score"] - mn) / (mx - mn) for it in items}

        dn = minmax_norm(dense)
        sn = minmax_norm(sparse)

        all_ids = set(dn) | set(sn)
        combined: List[Tuple[str, float]] = []
        for cid in all_ids:
            combined.append((cid, dn.get(cid, 0.0) + sn.get(cid, 0.0)))

        combined.sort(key=lambda x: x[1], reverse=True)
        top_ids = [cid for cid, _ in combined[:top_k]]

        by_id = {it["chunk_id"]: it for it in (dense + sparse)}
        return [by_id[cid] for cid in top_ids if cid in by_id]


# ============================================================================
# MULTI-SOURCE INDEXER (FIXED)
# ============================================================================

class MultiSourceIndexer:
    def __init__(self, conf: Dict[str, Any]):
        self.conf = conf
        self.chunk_size = conf["chunk_size"]
        self.chunk_overlap = conf["chunk_overlap"]
        self.docs = self.get_docs(conf["data_source"])

        self.indices: Dict[str, LocalIndex] = {}

        for doc in self.docs:
            doc_id = doc["doc_id"]
            index_dir = os.path.join(conf["indexes_base_dir"], doc_id)
            self.indices[doc_id] = LocalIndex(index_dir)

    def get_docs(self, source_path: str) -> List[dict]:
        docs = []
        for filename in sorted(os.listdir(source_path)):
            if not filename.lower().endswith(".txt"):
                continue
            file_path = os.path.join(source_path, filename)

            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()

            # FIX: Skippa file vuoti
            if len(text.strip()) == 0:
                print(f"⚠️  Skipping empty file: {filename}")
                continue

            doc_id = os.path.splitext(filename)[0]
            docs.append({
                "doc_id": doc_id,
                "title": doc_id,
                "path": file_path,
                "text": text
            })

        if not docs:
            raise FileNotFoundError("Nessun .txt valido trovato in " + source_path)

        return docs

    def doc2chunks(self, doc_text: str) -> List[str]:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            length_function=len,
            is_separator_regex=False,
        )
        return splitter.split_text(doc_text)

    def index_all(self) -> Dict[str, int]:
        vectorizer = Vectorizer(self.conf["embedder"])
        chunk_counts = {}

        print(f"\n🔧 Indicizzazione separata di {len(self.docs)} documenti...\n")

        for doc in tqdm(self.docs, desc="Indicizzazione documenti"):
            doc_id = doc["doc_id"]
            print(f"\n  📄 Processing: {doc_id}")

            # Chunking
            chunks = self.doc2chunks(doc["text"])

            # FIX: Se nessun chunk, skippa
            if len(chunks) == 0:
                print(f"      ⚠️  Nessun chunk creato, skippo indicizzazione")
                chunk_counts[doc_id] = 0
                continue

            chunk_records = []
            for chunk in chunks:
                # FIX: Skippa chunks vuoti
                if len(chunk.strip()) == 0:
                    continue

                chunk_id = doc_id + "_" + hashlib.md5(chunk.encode("utf-8")).hexdigest()
                chunk_records.append(
                    ChunkRecord(
                        chunk_id=chunk_id,
                        source_id=doc_id,
                        source_title=doc["title"],
                        source_path=doc["path"],
                        text=chunk
                    )
                )

            # FIX: Se nessun chunk valido, skippa
            if len(chunk_records) == 0:
                print(f"      ⚠️  Nessun chunk valido, skippo indicizzazione")
                chunk_counts[doc_id] = 0
                continue

            # Salva chunks
            index = self.indices[doc_id]
            index.add_chunks(chunk_records)
            index.save_chunks()
            index.save_conf(self.conf)

            # Embeddings
            texts = [c.text for c in chunk_records]
            emb = vectorizer.get_embeddings(texts)

            # Build FAISS
            index.build_faiss(emb)

            # Build BM25
            if self.conf.get("text_sim", "none") == "bm25":
                index.build_bm25()

            chunk_counts[doc_id] = len(chunk_records)
            print(f"     ✅ {len(chunk_records)} chunks creati")

        print(f"\n✅ Indicizzazione completata!")
        return chunk_counts


# ============================================================================
# SOURCE-AWARE RAG (FIXED)
# ============================================================================

class SourceAwareRag:
    def __init__(self, conf: Dict[str, Any]):
        self.conf = conf
        self.mode = conf["retrieval_mode"]
        self.include_metadata = conf["include_metadata"]

        # Carica tutti gli indici
        self.indices: Dict[str, LocalIndex] = {}

        indexes_base = conf["indexes_base_dir"]

        if not os.path.exists(indexes_base):
            raise ValueError(f"Directory indici non trovata: {indexes_base}")

        for source_name in os.listdir(indexes_base):
            index_dir = os.path.join(indexes_base, source_name)

            if not os.path.isdir(index_dir):
                continue

            if not os.path.exists(os.path.join(index_dir, "conf.json")):
                continue

            index = LocalIndex(index_dir)
            index.load()
            self.indices[source_name] = index

            # Info sul caricamento
            num_chunks = len(index._chunks)
            if num_chunks == 0:
                print(f"  ⚠️  Caricato indice VUOTO: {source_name} (0 chunks)")
            else:
                print(f"  ✅ Caricato indice: {source_name} ({num_chunks} chunks)")

        if not self.indices:
            raise ValueError(f"Nessun indice trovato in {indexes_base}")

        self.vectorizer = Vectorizer(conf["embedder"])

        self.template = ChatPromptTemplate([
            ("system",
             "Rispondi usando solo le informazioni nel contesto. "
             "Se mancano informazioni, dillo chiaramente."),
            ("human",
             "Domanda: {question}\n\nContesto:\n{context}\n\nRisposta:")
        ])

    def format_context(self, contexts: List[Dict[str, Any]]) -> str:
        if not contexts:
            return "[Nessun contesto disponibile]"

        blocks = []
        for c in contexts:
            if self.include_metadata:
                header = f"[Sorgente: {c.get('source_title','')} | Score: {c.get('score',0):.4f}]"
                blocks.append(header + "\n" + c["text"])
            else:
                blocks.append(c["text"])
        return "\n\n---\n\n".join(blocks)

    def retrieve(self, question: str, source: str) -> List[Dict[str, Any]]:
        """
        FIX: NO FALLBACK - Se source non trovata o vuota, ritorna lista vuota
        """
        # Normalizza source
        source_clean = source.replace('.txt', '').strip()

        # FIX: Se source non esiste, ritorna lista vuota (NO fallback)
        if source_clean not in self.indices:
            print(f"⚠️  Source '{source_clean}' non trovata negli indici disponibili: {list(self.indices.keys())}")
            print(f"   Ritorno lista vuota (no contexts).")
            return []

        # Prendi indice corretto
        index = self.indices[source_clean]

        # FIX: Se indice vuoto, ritorna lista vuota
        if len(index._chunks) == 0:
            print(f"⚠️  Indice '{source_clean}' è vuoto (0 chunks).")
            print(f"   Ritorno lista vuota (no contexts).")
            return []

        # Retrieval
        q_emb = self.vectorizer.get_embeddings(question)

        if self.mode == "dense":
            return index.dense_search(q_emb, top_k=self.conf["top_k"])
        elif self.mode == "hybrid":
            return index.hybrid_search(question, q_emb, top_k=self.conf["top_k"])
        else:
            raise ValueError("retrieval_mode non supportata: " + str(self.mode))

    def build_prompt(self, question: str, contexts: List[Dict[str, Any]]) -> str:
        context_str = self.format_context(contexts)
        return self.template.format(question=question, context=context_str)


# ============================================================================
# GENERATION FUNCTION (FIXED)
# ============================================================================

def run_rag_on_dataframe_with_source(
    df: pd.DataFrame,
    rag: SourceAwareRag,
    question_col: str = "Question",
    source_col: str = "Source",
    generate_func = None
) -> pd.DataFrame:
    """
    FIX: Gestisce correttamente source vuote/non trovate
    """

    if source_col not in df.columns:
        raise ValueError(f"Colonna '{source_col}' non trovata. Colonne: {list(df.columns)}")

    answers = []
    prompts = []
    top1_list = []
    topk_list = []
    sources_used = []
    num_contexts_list = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Generazione risposte RAG"):
        question = str(row[question_col])
        source = str(row[source_col])

        # Retrieve (può ritornare lista vuota se source non trovata/vuota)
        ctxs = rag.retrieve(question, source)

        num_contexts_list.append(len(ctxs))

        # FIX: Se nessun contesto, genera risposta che lo dice
        if len(ctxs) == 0:
            prompt = f"Domanda: {question}\n\nNessun contesto disponibile per la source '{source}'."
            if generate_func:
                ans = "Non posso rispondere: nessun contesto disponibile per questa fonte."
            else:
                ans = "[ERRORE: Nessun contesto trovato]"
        else:
            # Build prompt normale
            prompt = rag.build_prompt(question, ctxs)

            # Generate
            if generate_func:
                ans = generate_func(prompt)
            else:
                ans = "\n\n".join([c["text"] for c in ctxs[:3]])

        prompts.append(prompt)
        answers.append(ans)

        top_k_texts = [c["text"] for c in ctxs]
        topk_list.append("\n\n---\n\n".join(top_k_texts))
        top1_list.append(top_k_texts[0] if top_k_texts else "")
        sources_used.append(ctxs[0]["source_id"] if ctxs else source)

    out = df.copy()
    out["RAG_prompt"] = prompts
    out["RAG_answer"] = answers
    out["Top_k_contexts_rag"] = topk_list
    out["Top_1_context_rag"] = top1_list
    out["Source_used"] = sources_used
    out["Num_contexts"] = num_contexts_list  # FIX: Aggiungo colonna per debug

    return out


# ============================================================================
# CONFIGURATION
# ============================================================================

BASE_DIR = "/content/drive/MyDrive/ESAMI/AI"
TXT_DIR = os.path.join(BASE_DIR, "data_txt")
EXCEL_PATH = os.path.join(BASE_DIR, "solo_domande.xlsx")
INDEXES_BASE_DIR = os.path.join(BASE_DIR, "indexes_per_source")

INDEX_CONF = {
    "data_source": TXT_DIR,
    "indexes_base_dir": INDEXES_BASE_DIR,
    "chunk_size": 700,
    "chunk_overlap": 120,
    "embedder": "sentence-transformers/all-MiniLM-L6-v2",
    "text_sim": "bm25",
    "top_k": 5
}

RAG_CONF = {
    "indexes_base_dir": INDEXES_BASE_DIR,
    "embedder": INDEX_CONF["embedder"],
    "retrieval_mode": "hybrid",
    "include_metadata": True,
    "top_k": 5
}


# ============================================================================
# GROQ GENERATION FUNCTION
# ============================================================================

def generate_groq(prompt: str,
                  model: str = "llama-3.3-70b-versatile",
                  temperature: float = 0.2,
                  max_tokens: int = 200,
                  retries: int = 6) -> str:
    """
    Genera risposta usando Groq API con retry automatico su rate limits.
    """
    import requests
    import time
    import re

    def extract_retry_seconds(msg: str) -> float:
        m = re.search(r'try again in (\d+)m([\d.]+)s', msg)
        if m:
            minutes = int(m.group(1))
            seconds = float(m.group(2))
            return minutes * 60 + seconds

        m = re.search(r'try again in ([\d.]+)s', msg)
        if m:
            return float(m.group(1))

        return 60.0

    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"
    }
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    for attempt in range(retries):
        r = requests.post(url, headers=headers, json=payload)

        if r.status_code == 200:
            data = r.json()
            return data["choices"][0]["message"]["content"]

        # Rate limit
        if r.status_code == 429:
            try:
                msg = r.json()["error"]["message"]
            except Exception:
                msg = r.text

            wait_s = extract_retry_seconds(msg)
            wait_s += 0.2

            print(f"  ⏳ Rate limit hit. Waiting {wait_s:.1f}s (attempt {attempt + 1}/{retries})...")
            time.sleep(wait_s)
            continue

        # Altri errori
        raise RuntimeError(f"Errore Groq {r.status_code}: {r.text}")

    raise RuntimeError(f"Failed after {retries} retries")


# ============================================================================
# COMPLETE WORKFLOW EXAMPLE
# ============================================================================

def complete_workflow_example():
    """
    Workflow completo: indicizzazione → generazione → salvataggio
    """

    print("\n" + "="*80)
    print("RAG SOURCE-AWARE - WORKFLOW COMPLETO")
    print("="*80 + "\n")

    # ========================================
    # STEP 1: INDICIZZAZIONE (run once)
    # ========================================

    print("STEP 1: INDICIZZAZIONE SEPARATA PER SOURCE")
    print("-"*80)

    # Verifica se indici già esistono
    if os.path.exists(INDEXES_BASE_DIR) and os.listdir(INDEXES_BASE_DIR):
        print(f"\n⚠️  Indici già esistenti in {INDEXES_BASE_DIR}")

        user_input = input("Vuoi ricrearli da zero? (y/n): ").lower()

        if user_input == 'y':
            import shutil
            print("🗑️  Cancellazione vecchi indici...")
            shutil.rmtree(INDEXES_BASE_DIR)
            print("✅ Vecchi indici cancellati\n")

            # Crea nuovi indici
            indexer = MultiSourceIndexer(INDEX_CONF)
            chunk_counts = indexer.index_all()

            print("\n📊 Riepilogo indicizzazione:")
            for source, count in chunk_counts.items():
                print(f"  {source}: {count} chunks")
        else:
            print("✅ Uso indici esistenti\n")
    else:
        # Prima volta: crea indici
        indexer = MultiSourceIndexer(INDEX_CONF)
        chunk_counts = indexer.index_all()

        print("\n📊 Riepilogo indicizzazione:")
        for source, count in chunk_counts.items():
            print(f"  {source}: {count} chunks")


    # ========================================
    # STEP 2: CARICAMENTO DATI
    # ========================================

    print("\n" + "="*80)
    print("STEP 2: CARICAMENTO DATI")
    print("-"*80 + "\n")

    # Carica Excel
    if not os.path.exists(EXCEL_PATH):
        raise FileNotFoundError(f"File non trovato: {EXCEL_PATH}")

    df = pd.read_excel(EXCEL_PATH)
    print(f"✅ Dataset caricato: {len(df)} domande")
    print(f"   Colonne: {list(df.columns)}\n")

    # Verifica colonna Source
    if "Source" not in df.columns:
        print("⚠️  ATTENZIONE: Colonna 'Source' non trovata!")
        print("   Devi aggiungere la colonna Source al tuo Excel.")
        print("   Ogni riga deve indicare da quale .txt recuperare i contesti.\n")

        # Mostra esempio
        print("Esempio struttura Excel richiesta:")
        print("="*60)
        print("| Question                | Source      | Ground_truth |")
        print("|-------------------------|-------------|--------------|")
        print("| Quali prove...?         | primo_grado | Nel primo... |")
        print("| Cosa dice la sentenza...| appello     | La corte...  |")
        print("="*60 + "\n")

        return None

    print("✅ Colonna 'Source' trovata")
    print(f"\nDistribuzione sources:")
    for source, count in df['Source'].value_counts().items():
        print(f"  {source}: {count} domande")


    # ========================================
    # STEP 3: CARICAMENTO RAG
    # ========================================

    print("\n" + "="*80)
    print("STEP 3: CARICAMENTO RAG")
    print("-"*80 + "\n")

    rag = SourceAwareRag(RAG_CONF)
    print(f"\n✅ RAG caricato con {len(rag.indices)} indici separati\n")


    # ========================================
    # STEP 4: TEST RETRIEVAL (optional)
    # ========================================

    print("\n" + "="*80)
    print("STEP 4: TEST RETRIEVAL")
    print("-"*80 + "\n")

    # Test su prima domanda
    if len(df) > 0:
        test_row = df.iloc[0]
        test_question = str(test_row.get('Question', ''))
        test_source = str(test_row.get('Source', ''))

        print(f"Test question: {test_question[:60]}...")
        print(f"Source richiesta: {test_source}\n")

        contexts = rag.retrieve(test_question, test_source)

        print(f"📊 Recuperati {len(contexts)} contesti\n")

        if contexts:
            for i, ctx in enumerate(contexts[:3], 1):
                print(f"{i}. [{ctx['source_id']}] Score: {ctx['score']:.4f}")
                print(f"   {ctx['text'][:80]}...\n")
        else:
            print("⚠️  Nessun contesto recuperato (source vuota o non trovata)\n")


    # ========================================
    # STEP 5: GENERAZIONE RISPOSTE
    # ========================================

    print("\n" + "="*80)
    print("STEP 5: GENERAZIONE RISPOSTE")
    print("-"*80 + "\n")

    # Verifica API key Groq
    if "GROQ_API_KEY" not in os.environ:
        print("⚠️  GROQ_API_KEY non configurata!")
        print("   Imposta la chiave con: os.environ['GROQ_API_KEY'] = 'tua_chiave'")
        print("   Uso generazione dummy per testing...\n")

        def dummy_generate(prompt: str) -> str:
            return "[Risposta dummy - configura GROQ_API_KEY per generazione reale]"

        generate_func = dummy_generate
    else:
        print("✅ GROQ_API_KEY configurata")
        generate_func = generate_groq

    # Opzione: genera solo prime N domande per testing
    test_mode = input("\nGenerare tutte le domande? (y/n, default=y): ").lower()

    if test_mode == 'n':
        n_test = int(input("Quante domande vuoi testare? (default=3): ") or "3")
        df_test = df.head(n_test).copy()
        print(f"\n🔬 Modalità TEST: generazione di {n_test} domande\n")
    else:
        df_test = df.copy()
        print(f"\n🚀 Generazione di TUTTE le {len(df)} domande\n")

    # Genera risposte
    df_out = run_rag_on_dataframe_with_source(
        df=df_test,
        rag=rag,
        question_col="Question",
        source_col="Source",
        generate_func=generate_func
    )

    print("\n✅ Generazione completata!\n")


    # ========================================
    # STEP 6: VERIFICA RISULTATI
    # ========================================

    print("="*80)
    print("STEP 6: VERIFICA RISULTATI")
    print("-"*80 + "\n")

    # Statistiche
    print("📊 Statistiche:")
    print(f"  Totale domande: {len(df_out)}")
    print(f"  Domande con 0 contesti: {(df_out['Num_contexts'] == 0).sum()}")
    print(f"  Domande con >0 contesti: {(df_out['Num_contexts'] > 0).sum()}")

    # Distribuzione contesti per source
    print(f"\n📊 Contesti per source:")
    ctx_by_source = df_out.groupby('Source')['Num_contexts'].agg(['mean', 'min', 'max'])
    print(ctx_by_source)

    # Mostra prime 3 righe
    print(f"\n📄 Prime 3 righe risultati:")
    print(df_out[['Question', 'Source', 'Num_contexts', 'Source_used']].head(3))

    # Verifica source matching
    source_mismatch = df_out[df_out['Source'].str.replace('.txt', '').str.strip() != df_out['Source_used']]

    if len(source_mismatch) > 0:
        print(f"\n⚠️  ATTENZIONE: {len(source_mismatch)} domande con source mismatch:")
        print(source_mismatch[['Question', 'Source', 'Source_used', 'Num_contexts']])
    else:
        print(f"\n✅ Tutte le source corrispondono correttamente")


    # ========================================
    # STEP 7: SALVATAGGIO
    # ========================================

    print("\n" + "="*80)
    print("STEP 7: SALVATAGGIO RISULTATI")
    print("-"*80 + "\n")

    output_path = os.path.join(BASE_DIR, "dataset_with_source_aware_rag.xlsx")
    df_out.to_excel(output_path, index=False)

    print(f"💾 Risultati salvati in:")
    print(f"   {output_path}")

    print(f"\n✅ WORKFLOW COMPLETATO!")

    return df_out


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":

    # Esempio 1: Workflow completo interattivo
    print("\n" + "="*80)
    print("MODALITÀ DI ESECUZIONE")
    print("="*80 + "\n")
    print("1. Workflow completo interattivo")
    print("2. Solo indicizzazione")
    print("3. Solo generazione (indici già esistenti)")
    print("4. Debug mode")

    choice = input("\nScegli modalità (1-4, default=1): ").strip() or "1"

    if choice == "1":
        # Workflow completo
        df_result = complete_workflow_example()

    elif choice == "2":
        # Solo indicizzazione
        print("\n🔧 INDICIZZAZIONE")
        indexer = MultiSourceIndexer(INDEX_CONF)
        chunk_counts = indexer.index_all()

        print("\n📊 Riepilogo:")
        for source, count in chunk_counts.items():
            print(f"  {source}: {count} chunks")

    elif choice == "3":
        # Solo generazione
        print("\n🚀 GENERAZIONE RISPOSTE")

        # Carica dati
        df = pd.read_excel(EXCEL_PATH)

        # Carica RAG
        rag = SourceAwareRag(RAG_CONF)

        # Configura generazione
        if "GROQ_API_KEY" in os.environ:
            generate_func = generate_groq
        else:
            os.environ["GROQ_API_KEY"] = input("Inserisci GROQ_API_KEY: ")
            generate_func = generate_groq

        # Genera
        df_out = run_rag_on_dataframe_with_source(
            df=df,
            rag=rag,
            generate_func=generate_func
        )

        # Salva
        output_path = os.path.join(BASE_DIR, "dataset_with_source_aware_rag.xlsx")
        df_out.to_excel(output_path, index=False)
        print(f"\n💾 Salvato: {output_path}")

    elif choice == "4":
        # Debug mode
        print("\n🔍 DEBUG MODE")
        print("\nEsegui questi comandi per debug:\n")
        print("from debug_source_aware_rag import *")
        print("check_source_files(TXT_DIR)")
        print("check_index_structure(INDEXES_BASE_DIR)")
        print("check_dataframe_sources(EXCEL_PATH)")

    else:
        print("Scelta non valida")


# ============================================================================
# QUICK START GUIDE
# ============================================================================

"""
QUICK START - Copia e incolla questo codice nel tuo notebook:

```python
# 1. Setup
import os
os.environ["GROQ_API_KEY"] = "tua_chiave_qui"

from rag_source_aware_FIXED import *

# 2. Configura paths (opzionale, se diversi)
BASE_DIR = "/content/drive/MyDrive/ESAMI/AI"
TXT_DIR = os.path.join(BASE_DIR, "data_txt")
EXCEL_PATH = os.path.join(BASE_DIR, "solo_domande.xlsx")
INDEXES_BASE_DIR = os.path.join(BASE_DIR, "indexes_per_source")

# 3. Indicizza (SOLO PRIMA VOLTA)
indexer = MultiSourceIndexer(INDEX_CONF)
chunk_counts = indexer.index_all()

# 4. Carica dati
df = pd.read_excel(EXCEL_PATH)

# 5. Carica RAG
rag = SourceAwareRag(RAG_CONF)

# 6. Genera risposte
df_out = run_rag_on_dataframe_with_source(
    df=df,
    rag=rag,
    generate_func=generate_groq
)

# 7. Salva
df_out.to_excel("dataset_with_source_aware_rag.xlsx", index=False)
```

VERIFICA RISULTATI:
```python
# Check contesti per source
print(df_out.groupby('Source')['Num_contexts'].mean())

# Trova domande con 0 contesti (source vuote)
zero_ctx = df_out[df_out['Num_contexts'] == 0]
print(f"Domande con 0 contesti: {len(zero_ctx)}")
print(zero_ctx[['Question', 'Source', 'RAG_answer']])

# Verifica source matching
mismatch = df_out[df_out['Source'] != df_out['Source_used']]
if len(mismatch) > 0:
    print("⚠️ Source mismatch trovati!")
```

DEBUG (se hai problemi):
```python
from debug_source_aware_rag import *

# Verifica file sorgente
check_source_files(TXT_DIR)

# Verifica indici
check_index_structure(INDEXES_BASE_DIR)

# Test retrieval
test_retrieve_from_source(rag, "test question", "primo_grado")
```
"""


MODALITÀ DI ESECUZIONE

1. Workflow completo interattivo
2. Solo indicizzazione
3. Solo generazione (indici già esistenti)
4. Debug mode

Scegli modalità (1-4, default=1): 3

🚀 GENERAZIONE RISPOSTE
  ✅ Caricato indice: motivi_ricorso_cassazione_27_05_1991 (404 chunks)
  ✅ Caricato indice: sentenza_cassazione_12_02_1992 (481 chunks)
  ✅ Caricato indice: sentenza_processo_d_appello_18_07_1990 (1190 chunks)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generazione risposte RAG:   0%|          | 0/20 [00:00<?, ?it/s]

  ⏳ Rate limit hit. Waiting 2.5s (attempt 1/6)...
  ⏳ Rate limit hit. Waiting 3.8s (attempt 1/6)...
  ⏳ Rate limit hit. Waiting 5.0s (attempt 1/6)...
⚠️  Source 'sentenza_def_cassaz_23_11_1995' non trovata negli indici disponibili: ['motivi_ricorso_cassazione_27_05_1991', 'sentenza_cassazione_12_02_1992', 'sentenza_processo_d_appello_18_07_1990']
   Ritorno lista vuota (no contexts).
⚠️  Source 'sentenza_def_cassaz_23_11_1995' non trovata negli indici disponibili: ['motivi_ricorso_cassazione_27_05_1991', 'sentenza_cassazione_12_02_1992', 'sentenza_processo_d_appello_18_07_1990']
   Ritorno lista vuota (no contexts).
⚠️  Source 'sentenza_def_cassaz_23_11_1995' non trovata negli indici disponibili: ['motivi_ricorso_cassazione_27_05_1991', 'sentenza_cassazione_12_02_1992', 'sentenza_processo_d_appello_18_07_1990']
   Ritorno lista vuota (no contexts).
⚠️  Source 'sentenza_def_cassaz_23_11_1995' non trovata negli indici disponibili: ['motivi_ricorso_cassazione_27_05_1991', 'sentenza_cassaz

'\nQUICK START - Copia e incolla questo codice nel tuo notebook:\n\n```python\n# 1. Setup\nimport os\nos.environ["GROQ_API_KEY"] = "tua_chiave_qui"\n\nfrom rag_source_aware_FIXED import *\n\n# 2. Configura paths (opzionale, se diversi)\nBASE_DIR = "/content/drive/MyDrive/ESAMI/AI"\nTXT_DIR = os.path.join(BASE_DIR, "data_txt")\nEXCEL_PATH = os.path.join(BASE_DIR, "solo_domande.xlsx")\nINDEXES_BASE_DIR = os.path.join(BASE_DIR, "indexes_per_source")\n\n# 3. Indicizza (SOLO PRIMA VOLTA)\nindexer = MultiSourceIndexer(INDEX_CONF)\nchunk_counts = indexer.index_all()\n\n# 4. Carica dati\ndf = pd.read_excel(EXCEL_PATH)\n\n# 5. Carica RAG\nrag = SourceAwareRag(RAG_CONF)\n\n# 6. Genera risposte\ndf_out = run_rag_on_dataframe_with_source(\n    df=df,\n    rag=rag,\n    generate_func=generate_groq\n)\n\n# 7. Salva\ndf_out.to_excel("dataset_with_source_aware_rag.xlsx", index=False)\n```\n\nVERIFICA RISULTATI:\n```python\n# Check contesti per source\nprint(df_out.groupby(\'Source\')[\'Num_contexts\']

DA QUA IN POI USIAMO RAGAS

In [ ]:
df_out= pd.read_excel('/content/drive/MyDrive/ESAMI/AI/dataset_with_rag_topk.xlsx')

In [ ]:
pip install ragas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.8/358.8 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: jiter
    Found existing ins

DA QUA IN POI PER FARE ANALISI SUI RISULTATI DI RAGAS  

In [ ]:
#PER RICARICARE IRISULATI
import json

with open("/content/drive/MyDrive/ESAMI/AI/ragas_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)


Precision/Recall/F1 nel tuo codice non stanno confrontando la risposta con il contesto, ma con la Ground_truth.
Quindi misurano “quanto la risposta RAG copre e coincide con la ground truth”, claim per claim.
Faithfulness invece misura se le affermazioni della risposta sono supportate dal contesto recuperato (i top-k che hai passato).

l'esempio due è un'allucinazione. Indaghiamo cosa succede e che righe sono alluncinate

In [ ]:
ex = results[7]   # esempio 2
print("Domanda:", ex["Question"])
print("Precision:", ex["precision"])
print("Recall:", ex["recall"])
print("F1:", ex["f1"])
print("Faithfulness:", ex["faithfulness"])

pr = ex["pr_verdicts"]

print("\n=== PRECISION (Response claims vs Ground_truth) ===\n")
for i, v in enumerate(pr["response_vs_reference"], 1):
    status = "SUPPORTATO" if v["verdict"] else "NON SUPPORTATO"
    print(f"{i}. {status}")
    print("   Claim:", v["statement"])
    print("   Motivo:", v["reason"])
    print()


Domanda: La violazione di quali articoli viene eccepita nelle motivazioni del ricorso per cassazione in riferimento al proscioglimento del Picciafuoco?
Precision: 1.0
Recall: 0.6666666666666666
F1: 0.8
Faithfulness: 0.0

=== PRECISION (Response claims vs Ground_truth) ===

1. SUPPORTATO
   Claim: La violazione degli articoli 474 n° 4 C.p.p. (1980) viene eccepita nelle motivazioni del ricorso per cassazione.
   Motivo: The context explicitly mentions the violation of article 474 n° 4 C.p.p. (1980) in the motivations of the appeal for cassazione.

2. SUPPORTATO
   Claim: La violazione degli articoli 475 n° 3 C.p.p. (1980) viene eccepita nelle motivazioni del ricorso per cassazione.
   Motivo: The context explicitly mentions the violation of article 475 n° 3 C.p.p. (1980) in the motivations of the appeal for cassazione.

3. SUPPORTATO
   Claim: La violazione degli articoli 524 n° 3 C.p.p. (1980) viene eccepita nelle motivazioni del ricorso per cassazione.
   Motivo: The context explicitly

In [ ]:
fv = ex["faith_verdicts"]

print("\n=== FAITHFULNESS (Response claims vs Retrieved context) ===\n")
for i, v in enumerate(fv, 1):
    status = "SUPPORTATO" if v["verdict"] else "NON SUPPORTATO"
    print(f"{i}. {status}")
    print("   Statement:", v["statement"])
    if v["verdict"] == 1:
        print("   Motivo:", v["reason"])
    else:
        # qui v["reason"] è una lista di motivi (uno per documento del contesto)
        print("   Motivi (per documento):")
        for j, r in enumerate(v["reason"], 1):
            print(f"     [{j}] {r}")
    print()



=== FAITHFULNESS (Response claims vs Retrieved context) ===

1. NON SUPPORTATO
   Statement: La corte di cassazione non stabilisce nulla relativamente ai ricorsi presentati nei confronti di Fachini e Rinani.
   Motivi (per documento):
     [1] The context does not mention the outcome of the appeals presented against Fachini and Rinani.
     [2] The context explicitly states that the court of cassazione annulled the sentence of appeal due to its illogicità, lack of coerenza, and other flaws in the evaluation of the proofs and evidence.
     [3] The context does not mention the corte di cassazione stabilishing anything about the ricorsi presentati nei confronti di Fachini e Rinani.
     [4] The context explicitly states that the court declared some appeals inadmissible, but does not mention the outcome of the appeals against Fachini and Rinani.
     [5] The context does not mention the corte di cassazione making any statements about the ricorsi presentati nei confronti di Fachini e Rina

test solo sul LLM

In [7]:
import os
import pandas as pd
import requests

GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"

def groq_chat(messages, model="llama-3.3-70b-versatile", temperature=0.2, max_tokens=350, timeout=60):
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError("Imposta la variabile d'ambiente GROQ_API_KEY.")

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}",
    }
    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    r = requests.post(GROQ_URL, headers=headers, json=payload, timeout=timeout)
    if r.status_code != 200:
        raise RuntimeError(f"Errore Groq {r.status_code}: {r.text}")
    return r.json()["choices"][0]["message"]["content"]

def generate_answer(question: str, model="llama-3.3-70b-versatile") -> str:
    messages = [
        {"role": "system", "content": "Rispondi in modo accurato e conciso."},
        {"role": "user", "content": question},
    ]
    return groq_chat(messages, model=model, temperature=0.2, max_tokens=350)

# === INPUT/OUTPUT ===
INPUT_XLSX = "/content/drive/MyDrive/ESAMI/AI/solo_domande.xlsx"                 # <-- cambia
OUTPUT_XLSX = "/content/drive/MyDrive/ESAMI/AI/risposte_llm.xlsx"         # <-- cambia
MODEL = "llama-3.3-70b-versatile"

df = pd.read_excel(INPUT_XLSX)

if "Question" not in df.columns or "Ground_truth" not in df.columns:
    raise ValueError("Nel file Excel devono esserci le colonne: 'Question' e 'Ground_truth'.")

answers = []
for i, q in enumerate(df["Question"].astype(str).tolist(), 1):
    ans = generate_answer(q, model=MODEL)
    answers.append(ans)
    print(f"[{i}/{len(df)}] fatto")

df["Answer"] = answers
df["Answer_model"] = MODEL
df.to_excel(OUTPUT_XLSX, index=False)

print("Salvato:", OUTPUT_XLSX)


[1/20] fatto
[2/20] fatto
[3/20] fatto
[4/20] fatto
[5/20] fatto
[6/20] fatto
[7/20] fatto
[8/20] fatto
[9/20] fatto
[10/20] fatto
[11/20] fatto
[12/20] fatto
[13/20] fatto
[14/20] fatto
[15/20] fatto
[16/20] fatto
[17/20] fatto
[18/20] fatto
[19/20] fatto
[20/20] fatto
Salvato: /content/drive/MyDrive/ESAMI/AI/risposte_llm.xlsx


In [9]:
import os
import re
import json
import time
import pandas as pd
import requests

GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"

def groq_chat(messages, model="llama-3.3-70b-versatile", temperature=0.0, max_tokens=900, timeout=60):
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError("Imposta la variabile d'ambiente GROQ_API_KEY.")

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}",
    }
    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    r = requests.post(GROQ_URL, headers=headers, json=payload, timeout=timeout)
    if r.status_code != 200:
        raise RuntimeError(f"Errore Groq {r.status_code}: {r.text}")
    return r.json()["choices"][0]["message"]["content"]

def estrai_json_da_testo(s: str) -> str:
    s = s.strip()
    if (s.startswith("{") and s.endswith("}")) or (s.startswith("[") and s.endswith("]")):
        return s
    m = re.search(r"(\{.*\})", s, flags=re.DOTALL)
    if m:
        return m.group(1)
    m = re.search(r"(\[.*\])", s, flags=re.DOTALL)
    if m:
        return m.group(1)
    return s

def valuta_con_llm(question: str, answer: str, ground_truth: str, model="llama-3.3-70b-versatile") -> dict:
    schema = {
        "response_claims": ["..."],
        "reference_claims": ["..."],
        "response_vs_reference": [{"statement": "...", "verdict": True, "reason": "..."}],
        "reference_vs_response": [{"statement": "...", "verdict": True, "reason": "..."}],
    }

    messages = [
        {
            "role": "system",
            "content": (
                "Sei un valutatore rigoroso. Rispondi SOLO con JSON valido.\n"
                "- Scomponi RISPOSTA e GROUND_TRUTH in affermazioni atomiche.\n"
                "- response_vs_reference: ogni affermazione della risposta è true SOLO se il ground truth la supporta chiaramente.\n"
                "- reference_vs_response: ogni affermazione del ground truth è true SOLO se la risposta la copre chiaramente.\n"
                "- Se ambiguo o mancano dettagli: false.\n"
                f"Schema atteso: {json.dumps(schema, ensure_ascii=False)}"
            ),
        },
        {
            "role": "user",
            "content": (
                f"DOMANDA:\n{question}\n\n"
                f"RISPOSTA:\n{answer}\n\n"
                f"GROUND_TRUTH:\n{ground_truth}\n\n"
                "Produci il JSON."
            ),
        },
    ]

    txt = groq_chat(messages, model=model, temperature=0.0, max_tokens=900)
    js = estrai_json_da_testo(txt)
    return json.loads(js)

def calcola_metriche(verdicts: dict):
    rvr = verdicts.get("response_vs_reference", []) or []
    rrs = verdicts.get("reference_vs_response", []) or []

    tot_resp = len(rvr)
    sup_resp = sum(1 for x in rvr if bool(x.get("verdict")))
    tot_ref = len(rrs)
    cov_ref = sum(1 for x in rrs if bool(x.get("verdict")))

    precision = sup_resp / tot_resp if tot_resp else 0.0
    recall = cov_ref / tot_ref if tot_ref else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return precision, recall, f1

# === INPUT/OUTPUT ===
INPUT_XLSX = "/content/drive/MyDrive/ESAMI/AI/risposte_llm.xlsx"          # <-- deve contenere Answer
OUTPUT_XLSX = "/content/drive/MyDrive/ESAMI/AI/ragas_llm.xlsx"
JUDGE_MODEL = "llama-3.3-70b-versatile"

df = pd.read_excel(INPUT_XLSX)

required = {"Question", "Ground_truth", "Answer"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Mancano colonne: {missing}. Assicurati di aver eseguito la cella 1.")

precisions, recalls, f1s, verdicts_col = [], [], [], []

MAX_RETRIES = 3
SLEEP_SEC = 2

for i, row in df.iterrows():
    q = str(row["Question"])
    gt = str(row["Ground_truth"])
    ans = str(row["Answer"])

    last_err = None
    verdicts = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            verdicts = valuta_con_llm(q, ans, gt, model=JUDGE_MODEL)
            break
        except Exception as e:
            last_err = e
            if attempt < MAX_RETRIES:
                time.sleep(SLEEP_SEC)

    if verdicts is None:
        precisions.append(None)
        recalls.append(None)
        f1s.append(None)
        verdicts_col.append(json.dumps({"error": str(last_err)}, ensure_ascii=False))
    else:
        p, r, f1 = calcola_metriche(verdicts)
        precisions.append(p)
        recalls.append(r)
        f1s.append(f1)
        verdicts_col.append(json.dumps(verdicts, ensure_ascii=False))

    print(f"[{i+1}/{len(df)}] valutato")

df["precision"] = precisions
df["recall"] = recalls
df["f1"] = f1s
df["judge_model"] = JUDGE_MODEL
df["pr_verdicts_json"] = verdicts_col

df.to_excel(OUTPUT_XLSX, index=False)
print("Salvato:", OUTPUT_XLSX)


[1/20] valutato
[2/20] valutato
[3/20] valutato
[4/20] valutato
[5/20] valutato
[6/20] valutato
[7/20] valutato
[8/20] valutato
[9/20] valutato
[10/20] valutato
[11/20] valutato
[12/20] valutato
[13/20] valutato
[14/20] valutato
[15/20] valutato
[16/20] valutato
[17/20] valutato
[18/20] valutato
[19/20] valutato
[20/20] valutato
Salvato: /content/drive/MyDrive/ESAMI/AI/ragas_llm.xlsx


In [7]:
"""
Pipeline completa: Generazione risposte LLM + Valutazione RAGAS-like
====================================================================
1. Genera risposte con LLM (solo Question come input)
2. Valuta con RAGAS-like (confronto Answer vs Ground_truth)
3. Salva Excel con risposte + JSON con metriche dettagliate
"""

import os
import json
import re
import time
import pandas as pd
import requests
from typing import Dict, List, Any


# ============================================================================
# CONFIGURAZIONE API GROQ
# ============================================================================

GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"

def groq_chat(messages, model="llama-3.3-70b-versatile", temperature=0.0, max_tokens=900, timeout=60):
    """Chiamata API Groq"""
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError("Imposta la variabile d'ambiente GROQ_API_KEY.")

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}",
    }
    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    r = requests.post(GROQ_URL, headers=headers, json=payload, timeout=timeout)
    if r.status_code != 200:
        raise RuntimeError(f"Errore Groq {r.status_code}: {r.text}")
    return r.json()["choices"][0]["message"]["content"]


# ============================================================================
# STEP 1: GENERAZIONE RISPOSTE LLM
# ============================================================================

def generate_answer(question: str, model="llama-3.3-70b-versatile") -> str:
    """
    Genera risposta alla domanda usando solo l'LLM (senza contesto).

    Args:
        question: La domanda
        model: Modello da usare

    Returns:
        La risposta generata
    """
    messages = [
        {
            "role": "system",
            "content": "Sei un assistente esperto in storia italiana. Rispondi in modo accurato, conciso e basandoti sulle tue conoscenze. Se non sei sicuro di una informazione, indicalo."
        },
        {
            "role": "user",
            "content": question
        },
    ]
    return groq_chat(messages, model=model, temperature=0.2, max_tokens=350)


# ============================================================================
# STEP 2: VALUTAZIONE RAGAS-LIKE
# ============================================================================

def estrai_json_da_testo(s: str) -> str:
    """Estrae JSON da testo che potrebbe contenere altro"""
    s = s.strip()
    if (s.startswith("{") and s.endswith("}")) or (s.startswith("[") and s.endswith("]")):
        return s

    # Cerca pattern JSON
    m = re.search(r"(\{.*\})", s, flags=re.DOTALL)
    if m:
        return m.group(1)
    m = re.search(r"(\[.*\])", s, flags=re.DOTALL)
    if m:
        return m.group(1)
    return s


def valuta_con_llm(question: str, answer: str, ground_truth: str, model="llama-3.3-70b-versatile") -> dict:
    """
    Valuta la risposta confrontandola con la ground truth usando LLM as judge.

    Args:
        question: La domanda
        answer: La risposta generata dall'LLM
        ground_truth: La risposta corretta
        model: Modello judge

    Returns:
        Dict con claims e verdicts
    """
    schema = {
        "response_claims": ["claim1", "claim2", "..."],
        "reference_claims": ["claim1", "claim2", "..."],
        "response_vs_reference": [
            {"statement": "claim dalla risposta", "verdict": True, "reason": "motivo"}
        ],
        "reference_vs_response": [
            {"statement": "claim dalla ground truth", "verdict": True, "reason": "motivo"}
        ],
    }

    messages = [
        {
            "role": "system",
            "content": (
                "Sei un valutatore rigoroso che deve analizzare se una risposta è corretta.\n\n"
                "COMPITO:\n"
                "1. Scomponi RISPOSTA e GROUND_TRUTH in affermazioni atomiche (claims)\n"
                "2. Valuta ogni claim della risposta contro la ground truth (response_vs_reference)\n"
                "3. Valuta ogni claim della ground truth contro la risposta (reference_vs_response)\n\n"
                "REGOLE DI VALUTAZIONE:\n"
                "- verdict = true SOLO se l'affermazione è CHIARAMENTE supportata dall'altro testo\n"
                "- verdict = false se: mancano dettagli, ci sono contraddizioni, informazioni ambigue\n"
                "- Sii preciso con date, nomi, numeri: anche piccole differenze = false\n"
                "- Nel reason spiega brevemente perché hai dato quel verdict\n\n"
                "IMPORTANTE: Rispondi SOLO con JSON valido, nessun altro testo.\n\n"
                f"Schema atteso:\n{json.dumps(schema, ensure_ascii=False, indent=2)}"
            ),
        },
        {
            "role": "user",
            "content": (
                f"DOMANDA:\n{question}\n\n"
                f"RISPOSTA GENERATA:\n{answer}\n\n"
                f"GROUND_TRUTH (risposta corretta):\n{ground_truth}\n\n"
                "Produci il JSON con la valutazione completa."
            ),
        },
    ]

    txt = groq_chat(messages, model=model, temperature=0.0, max_tokens=900)
    js = estrai_json_da_testo(txt)
    return json.loads(js)


def calcola_metriche(verdicts: dict) -> tuple:
    """
    Calcola precision, recall, F1 dai verdicts.

    Args:
        verdicts: Dict con response_vs_reference e reference_vs_response

    Returns:
        (precision, recall, f1)
    """
    rvr = verdicts.get("response_vs_reference", []) or []
    rrs = verdicts.get("reference_vs_response", []) or []

    # Precision: quante affermazioni della risposta sono supportate dalla GT
    tot_resp = len(rvr)
    sup_resp = sum(1 for x in rvr if bool(x.get("verdict")))
    precision = sup_resp / tot_resp if tot_resp else 0.0

    # Recall: quante affermazioni della GT sono coperte dalla risposta
    tot_ref = len(rrs)
    cov_ref = sum(1 for x in rrs if bool(x.get("verdict")))
    recall = cov_ref / tot_ref if tot_ref else 0.0

    # F1
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    return precision, recall, f1


# ============================================================================
# PIPELINE COMPLETA
# ============================================================================

def run_llm_generation_and_evaluation(
    input_excel: str,
    output_excel: str,
    output_json: str,
    generation_model: str = "llama-3.3-70b-versatile",
    judge_model: str = "llama-3.3-70b-versatile",
    max_retries: int = 3,
    sleep_sec: int = 2
):
    """
    Pipeline completa:
    1. Genera risposte con LLM
    2. Valuta con RAGAS-like
    3. Salva risultati

    Args:
        input_excel: Path file Excel con Question e Ground_truth
        output_excel: Path output Excel con risposte e metriche
        output_json: Path output JSON con dettagli completi
        generation_model: Modello per generare risposte
        judge_model: Modello per valutare
        max_retries: Tentativi massimi per chiamate API
        sleep_sec: Secondi di attesa tra retry
    """

    print("="*80)
    print("PIPELINE LLM GENERATION + RAGAS EVALUATION")
    print("="*80)

    # 1) Carica dati
    print(f"\n📂 Caricamento dati da: {input_excel}")
    df = pd.read_excel(input_excel)

    # Verifica colonne
    required = {"Question", "Ground_truth"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Mancano colonne: {missing}")

    print(f"✅ Trovate {len(df)} domande\n")

    # 2) Inizializza liste risultati
    answers = []
    precisions = []
    recalls = []
    f1s = []
    verdicts_col = []

    # 3) Processa ogni domanda
    print("="*80)
    print("STEP 1: GENERAZIONE RISPOSTE")
    print("="*80)

    for i, row in df.iterrows():
        question = str(row["Question"])

        print(f"\n[{i+1}/{len(df)}] Generazione risposta...")

        # Genera risposta con retry
        last_err = None
        answer = None
        for attempt in range(1, max_retries + 1):
            try:
                answer = generate_answer(question, model=generation_model)
                break
            except Exception as e:
                last_err = e
                print(f"   ⚠️ Tentativo {attempt}/{max_retries} fallito: {e}")
                if attempt < max_retries:
                    time.sleep(sleep_sec)

        if answer is None:
            answer = f"[ERRORE GENERAZIONE: {last_err}]"

        answers.append(answer)
        print(f"   ✅ Risposta generata ({len(answer)} caratteri)")

    # Aggiungi risposte al DataFrame
    df["Answer"] = answers
    df["Answer_model"] = generation_model

    print("\n" + "="*80)
    print("STEP 2: VALUTAZIONE RAGAS-LIKE")
    print("="*80)

    # 4) Valuta ogni risposta
    for i, row in df.iterrows():
        question = str(row["Question"])
        ground_truth = str(row["Ground_truth"])
        answer = str(row["Answer"])

        print(f"\n[{i+1}/{len(df)}] Valutazione risposta...")

        # Salta se la generazione è fallita
        if answer.startswith("[ERRORE GENERAZIONE"):
            precisions.append(None)
            recalls.append(None)
            f1s.append(None)
            verdicts_col.append(json.dumps({"error": "Generazione fallita"}, ensure_ascii=False))
            print(f"   ⚠️ Saltata (generazione fallita)")
            continue

        # Valuta con retry
        last_err = None
        verdicts = None
        for attempt in range(1, max_retries + 1):
            try:
                verdicts = valuta_con_llm(question, answer, ground_truth, model=judge_model)
                break
            except Exception as e:
                last_err = e
                print(f"   ⚠️ Tentativo {attempt}/{max_retries} fallito: {e}")
                if attempt < max_retries:
                    time.sleep(sleep_sec)

        if verdicts is None:
            precisions.append(None)
            recalls.append(None)
            f1s.append(None)
            verdicts_col.append(json.dumps({"error": str(last_err)}, ensure_ascii=False))
            print(f"   ❌ Valutazione fallita")
        else:
            p, r, f1 = calcola_metriche(verdicts)
            precisions.append(p)
            recalls.append(r)
            f1s.append(f1)
            verdicts_col.append(json.dumps(verdicts, ensure_ascii=False))
            print(f"   ✅ P={p:.3f}, R={r:.3f}, F1={f1:.3f}")

    # 5) Aggiungi metriche al DataFrame
    df["precision"] = precisions
    df["recall"] = recalls
    df["f1"] = f1s
    df["judge_model"] = judge_model
    df["pr_verdicts_json"] = verdicts_col

    # 6) Salva Excel
    print("\n" + "="*80)
    print("SALVATAGGIO RISULTATI")
    print("="*80)

    df.to_excel(output_excel, index=False)
    print(f"\n✅ Excel salvato: {output_excel}")

    # 7) Salva JSON dettagliato
    results_json = []
    for i, row in df.iterrows():
        try:
            verdicts = json.loads(row["pr_verdicts_json"]) if pd.notna(row["pr_verdicts_json"]) else {}
        except:
            verdicts = {}

        results_json.append({
            "Question": row["Question"],
            "Answer": row.get("Answer", ""),
            "Ground_truth": row["Ground_truth"],
            "precision": row.get("precision"),
            "recall": row.get("recall"),
            "f1": row.get("f1"),
            "pr_verdicts": verdicts
        })

    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(results_json, f, ensure_ascii=False, indent=2)
    print(f"✅ JSON salvato: {output_json}")

    # 8) Statistiche finali
    print("\n" + "="*80)
    print("STATISTICHE FINALI")
    print("="*80)

    valid_f1 = [f for f in f1s if f is not None]
    if valid_f1:
        print(f"\nF1 Score medio: {sum(valid_f1)/len(valid_f1):.3f}")
        print(f"F1 Score min:   {min(valid_f1):.3f}")
        print(f"F1 Score max:   {max(valid_f1):.3f}")

    valid_precision = [p for p in precisions if p is not None]
    if valid_precision:
        print(f"\nPrecision media: {sum(valid_precision)/len(valid_precision):.3f}")

    valid_recall = [r for r in recalls if r is not None]
    if valid_recall:
        print(f"Recall media:    {sum(valid_recall)/len(valid_recall):.3f}")

    errors = sum(1 for a in answers if a.startswith("[ERRORE"))
    print(f"\nErrori totali: {errors}/{len(df)}")

    print("\n" + "="*80)
    print("✅ PIPELINE COMPLETATA!")
    print("="*80)


# ============================================================================
# ESECUZIONE
# ============================================================================

if __name__ == "__main__":
    # Configurazione paths
    INPUT_EXCEL = "/content/drive/MyDrive/ESAMI/AI/solo_domande.xlsx"
    OUTPUT_EXCEL = "/content/drive/MyDrive/ESAMI/AI/llm_risposte_e_metriche_2.xlsx"
    OUTPUT_JSON = "/content/drive/MyDrive/ESAMI/AI/llm_ragas_results_2.json"

    # Configurazione modelli
    GENERATION_MODEL = "llama-3.3-70b-versatile"
    JUDGE_MODEL = "llama-3.3-70b-versatile"

    # Esegui pipeline
    run_llm_generation_and_evaluation(
        input_excel=INPUT_EXCEL,
        output_excel=OUTPUT_EXCEL,
        output_json=OUTPUT_JSON,
        generation_model=GENERATION_MODEL,
        judge_model=JUDGE_MODEL,
        max_retries=3,
        sleep_sec=2
    )

PIPELINE LLM GENERATION + RAGAS EVALUATION

📂 Caricamento dati da: /content/drive/MyDrive/ESAMI/AI/solo_domande.xlsx
✅ Trovate 20 domande

STEP 1: GENERAZIONE RISPOSTE

[1/20] Generazione risposta...
   ✅ Risposta generata (252 caratteri)

[2/20] Generazione risposta...
   ✅ Risposta generata (217 caratteri)

[3/20] Generazione risposta...
   ✅ Risposta generata (1084 caratteri)

[4/20] Generazione risposta...
   ✅ Risposta generata (1251 caratteri)

[5/20] Generazione risposta...
   ✅ Risposta generata (1375 caratteri)

[6/20] Generazione risposta...
   ✅ Risposta generata (369 caratteri)

[7/20] Generazione risposta...
   ✅ Risposta generata (1298 caratteri)

[8/20] Generazione risposta...
   ✅ Risposta generata (1059 caratteri)

[9/20] Generazione risposta...
   ✅ Risposta generata (1286 caratteri)

[10/20] Generazione risposta...
   ✅ Risposta generata (340 caratteri)

[11/20] Generazione risposta...
   ✅ Risposta generata (402 caratteri)

[12/20] Generazione risposta...
   ✅ Rispo

In [ ]:
-------------------------------------------------------------------------------
CLAUDE PER GENRARE JSON RAGAS

In [ ]:
# ==========================
# UTILITÀ: TAGLIO TESTI
# ==========================

def tronca_testo(x, max_caratteri):
    """Tronca il testo a un massimo di caratteri."""
    s = "" if x is None else str(x)
    s = s.strip()
    return s[:max_caratteri]


def tronca_lista_testi(lista, max_caratteri):
    """Tronca ogni elemento della lista."""
    return [tronca_testo(t, max_caratteri) for t in lista if str(t).strip()]


def sanitize_text_for_json(text):
    """Rimuove caratteri problematici per il JSON."""
    if text is None:
        return ""
    # Normalizza apostrofi e virgolette
    text = str(text).replace("'", "'").replace(""", '"').replace(""", '"')
    # Rimuove caratteri di controllo
    text = "".join(char for char in text if ord(char) >= 32 or char in '\n\t')
    return text.strip()


# ==========================
# PARAMETRI DI CONTROLLO
# ==========================

MAX_CLAIMS = 20                 # limite massimo di asserzioni per risposta/ground truth
MAX_REF_CHARS = 5000            # taglio ground truth
MAX_RESP_CHARS = 5000           # taglio risposta RAG
MAX_DOC_CHARS = 6000            # taglio per singolo documento di contesto
MAX_DOCS = None                 # se vuoi limitarli: es. 5 (altrimenti lascia None)
MAX_RETRIES = 3                 # numero di tentativi per chiamate API fallite


# ==========================
# FUNZIONI ASYNC CON RETRY
# ==========================

async def extract_claims_with_retry(text, atomicity="low", coverage="low", max_claims=MAX_CLAIMS, max_retries=MAX_RETRIES):
    """Estrae claims con retry in caso di errore."""
    # Sanitizza il testo in input
    text = sanitize_text_for_json(text)

    client = AsyncOpenAI(
        api_key=os.environ["GROQ_API_KEY"],
        base_url="https://api.groq.com/openai/v1"
    )
    llm = llm_factory("llama-3.3-70b-versatile", client=client)
    scorer = FactualCorrectness(llm=llm, atomicity=atomicity, coverage=coverage)

    for attempt in range(max_retries):
        try:
            claims = await scorer._decompose_claims(text)
            if claims is None:
                return []
            return claims[:max_claims]
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"   ⚠ Estrazione claims fallita dopo {max_retries} tentativi: {repr(e)}")
                return []  # Ritorna lista vuota invece di fallire
            print(f"   Tentativo {attempt + 1}/{max_retries} fallito, riprovo...")
            await asyncio.sleep(1)  # Pausa breve prima di riprovare


async def verify_claims_with_retry(scorer, claims, context_text, max_retries=MAX_RETRIES):
    """Verifica claims con retry in caso di errore."""
    # Sanitizza il contesto
    context_text = sanitize_text_for_json(context_text)

    for attempt in range(max_retries):
        try:
            return await scorer._verify_claims(claims, context_text)
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"   ⚠ Verifica claims fallita dopo {max_retries} tentativi: {repr(e)}")
                return None
            print(f"   Tentativo {attempt + 1}/{max_retries} fallito, riprovo...")
            await asyncio.sleep(1)


async def evaluate_precision_recall_f1(response_claims, reference_claims, response_text, reference_text):
    """Calcola precision, recall e F1 con gestione errori."""
    client = AsyncOpenAI(
        api_key=os.environ["GROQ_API_KEY"],
        base_url="https://api.groq.com/openai/v1"
    )
    llm = llm_factory("llama-3.3-70b-versatile", client=client, max_tokens=2048)
    scorer = FactualCorrectness(llm=llm)

    # Verifica response claims vs reference
    resp_ref = await verify_claims_with_retry(scorer, response_claims, reference_text)
    if resp_ref is None or not hasattr(resp_ref, 'statements'):
        # Fallback in caso di errore
        return {
            "tp": 0, "fp": 0, "fn": 0,
            "precision": 0.0, "recall": 0.0, "f1": 0.0
        }, {
            "response_vs_reference": [],
            "reference_vs_response": []
        }

    tp = sum(v.verdict for v in resp_ref.statements)
    fp = len(resp_ref.statements) - tp

    # Verifica reference claims vs response
    ref_resp = await verify_claims_with_retry(scorer, reference_claims, response_text)
    if ref_resp is None or not hasattr(ref_resp, 'statements'):
        fn = len(reference_claims)  # Fallback conservativo
        ref_resp_verdicts = []
    else:
        fn = sum(not v.verdict for v in ref_resp.statements)
        ref_resp_verdicts = [
            {"statement": v.statement, "verdict": int(v.verdict), "reason": v.reason}
            for v in ref_resp.statements
        ]

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    metrics = {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}

    verdicts_dict = {
        "response_vs_reference": [
            {"statement": v.statement, "verdict": int(v.verdict), "reason": v.reason}
            for v in resp_ref.statements
        ],
        "reference_vs_response": ref_resp_verdicts
    }

    return metrics, verdicts_dict


async def evaluate_faithfulness_multi(statements, context):
    """Valuta faithfulness su multipli documenti con gestione errori."""
    client = AsyncOpenAI(
        api_key=os.environ["GROQ_API_KEY"],
        base_url="https://api.groq.com/openai/v1"
    )
    llm = llm_factory("llama-3.3-70b-versatile", client=client, max_tokens=4096)
    scorer = Faithfulness(llm=llm)

    per_statement = defaultdict(list)

    for doc in context:
        # Sanitizza ogni documento
        doc = sanitize_text_for_json(doc)

        for attempt in range(MAX_RETRIES):
            try:
                verdicts = await scorer._create_verdicts(statements, doc)

                for v in verdicts.statements:
                    per_statement[v.statement].append({
                        "verdict": int(v.verdict),
                        "reason": v.reason,
                    })
                break  # Successo, esci dal retry loop

            except Exception as e:
                if attempt == MAX_RETRIES - 1:
                    print(f"   ⚠ Valutazione faithfulness fallita per un documento: {repr(e)}")
                    # Continua con il prossimo documento
                else:
                    await asyncio.sleep(1)

    # Aggrega i verdetti
    aggregated_verdicts = []

    for statement, verdicts in per_statement.items():
        supporting = [v for v in verdicts if v["verdict"] == 1]
        non_supporting = [v for v in verdicts if v["verdict"] == 0]

        if supporting:
            aggregated_verdicts.append({
                "statement": statement,
                "verdict": 1,
                "reason": supporting[0]["reason"],
            })
        else:
            aggregated_verdicts.append({
                "statement": statement,
                "verdict": 0,
                "reason": [v["reason"] for v in non_supporting] if non_supporting else ["Nessun documento di supporto"],
            })

    supported = sum(v["verdict"] for v in aggregated_verdicts)
    total = len(aggregated_verdicts)
    score = supported / total if total > 0 else 0.0

    return score, aggregated_verdicts


# ==========================
# ELABORAZIONE DATASET
# ==========================

async def run_one_row(row):
    """Processa una singola riga del dataset."""
    question = sanitize_text_for_json(row["Question"])

    # Taglio ground truth e risposta
    reference = tronca_testo(row["Ground_truth"], MAX_REF_CHARS)
    response = tronca_testo(row["RAG_answer"], MAX_RESP_CHARS)

    # Split contesti, pulizia e taglio
    contexts_raw = str(row["Top_k_contexts_rag"]).split("\n\n---\n\n")
    all_docs_context = tronca_lista_testi(contexts_raw, MAX_DOC_CHARS)

    # Limite sul numero di documenti
    if MAX_DOCS is not None:
        all_docs_context = all_docs_context[:MAX_DOCS]

    # 1) Estrazione asserzioni (con retry)
    reference_claims = await extract_claims_with_retry(reference, max_claims=MAX_CLAIMS)
    response_claims = await extract_claims_with_retry(response, max_claims=MAX_CLAIMS)

    # Se entrambe le liste sono vuote, salta la valutazione
    if not reference_claims and not response_claims:
        print("   ⚠ Nessuna asserzione estratta, salto valutazione")
        return {
            "Question": question,
            "precision": None,
            "recall": None,
            "f1": None,
            "faithfulness": None,
            "tp": None,
            "fp": None,
            "fn": None,
            "pr_verdicts": None,
            "faith_verdicts": None,
            "warning": "Nessuna asserzione estratta"
        }

    # 2) Precision/Recall/F1
    pr_metrics, pr_verdicts = await evaluate_precision_recall_f1(
        response_claims, reference_claims, response, reference
    )

    # 3) Faithfulness (solo se ci sono response claims)
    if response_claims:
        faith_score, faith_verdicts = await evaluate_faithfulness_multi(
            response_claims, all_docs_context
        )
    else:
        faith_score, faith_verdicts = 0.0, []

    return {
        "Question": question,
        "precision": pr_metrics["precision"],
        "recall": pr_metrics["recall"],
        "f1": pr_metrics["f1"],
        "faithfulness": faith_score,
        "tp": pr_metrics["tp"],
        "fp": pr_metrics["fp"],
        "fn": pr_metrics["fn"],
        "pr_verdicts": pr_verdicts,
        "faith_verdicts": faith_verdicts
    }


async def run_all(df, save_path="ragas_results.json"):
    """
    Processa tutte le righe del dataset con salvataggio progressivo.
    Riprende da dove si era fermato se il file esiste già.
    """
    # 1) Carica risultati esistenti se presenti
    if os.path.exists(save_path):
        with open(save_path, "r", encoding="utf-8") as f:
            results = json.load(f)
        print(f"Trovati {len(results)} risultati esistenti, riprendo da lì...")
    else:
        results = []

    start_idx = len(results)
    total = len(df)

    if start_idx >= total:
        print("Tutti gli esempi sono già stati calcolati!")
        return results

    print(f"\nElaborazione da esempio {start_idx + 1} a {total}...")

    # 2) Processa ogni riga rimanente
    for i in range(start_idx, total):
        row = df.iloc[i]
        print(f"\n===== ESEMPIO {i+1}/{total} =====")

        try:
            out = await run_one_row(row)

            # Mostra risultati
            if out["precision"] is not None:
                print(f"Precision: {out['precision']:.3f}, "
                      f"Recall: {out['recall']:.3f}, "
                      f"F1: {out['f1']:.3f}")
                print(f"Faithfulness: {out['faithfulness']:.3f}")
            else:
                print("Risultati parziali salvati con warning")

            results.append(out)

        except Exception as e:
            # Salva entry con errore e continua
            print(f"\n*** ERRORE nell'esempio {i+1}: {repr(e)} ***")
            print("Salvo risultato parziale e continuo...")

            results.append({
                "Question": sanitize_text_for_json(row["Question"]),
                "precision": None,
                "recall": None,
                "f1": None,
                "faithfulness": None,
                "tp": None,
                "fp": None,
                "fn": None,
                "pr_verdicts": None,
                "faith_verdicts": None,
                "error": repr(e),
                "row_index": int(i)
            })

        # 3) Salva dopo ogni esempio (sovrascrive con tutti i risultati accumulati)
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"\n✓ Completato! {len(results)} risultati salvati in '{save_path}'")
    return results

In [ ]:
df_out= pd.read_excel('/content/drive/MyDrive/ESAMI/AI/dataset_with_source_aware_rag.xlsx')

In [ ]:
from openai import AsyncOpenAI
import os
import pandas as pd
import requests
import json
import hashlib
from typing import List, Dict
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List, Union
import numpy as np
from langchain_core.prompts import ChatPromptTemplate
from __future__ import annotations
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple
import pickle
import numpy as np
import time, re
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import FactualCorrectness
import asyncio
from ragas.metrics.collections import Faithfulness
from collections import defaultdict
from collections import defaultdict

ModuleNotFoundError: No module named 'langchain_text_splitters'

In [ ]:
import numpy as np
import time, re
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import FactualCorrectness
import asyncio
from ragas.metrics.collections import Faithfulness
from collections import defaultdict
from collections import defaultdict

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
import asyncio
# ... carica il tuo dataframe df ...

results = await run_all(df_out, save_path="/content/drive/MyDrive/ESAMI/AI/ragas_results_70_ok.json")

Trovati 18 risultati esistenti, riprendo da lì...

Elaborazione da esempio 19 a 20...

===== ESEMPIO 19/20 =====
Precision: 0.000, Recall: 0.000, F1: 0.000
Faithfulness: 0.500

===== ESEMPIO 20/20 =====
Precision: 0.000, Recall: 0.000, F1: 0.000
Faithfulness: 0.500

✓ Completato! 20 risultati salvati in '/content/drive/MyDrive/ESAMI/AI/ragas_results_70_ok.json'


In [3]:
"""
Analisi Posizione FP/FN - CON SIMILARITÀ SEMANTICA AI CONTESTI
===============================================================
Per i False Positives, calcola:
1. Similarità tra FP e ground truth
2. Similarità tra FP e tutti i contesti (Top_k)
3. Similarità tra FP e il miglior contesto (Top_1)
"""

import json
import re
import pandas as pd
import numpy as np
from typing import List, Dict, Any, Tuple
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================================
# SENTENCE SPLITTING
# ============================================================================

_SENT_SPLIT = re.compile(r'(?<=[.!?])\s+')

def split_sentences(text: str) -> List[str]:
    """Split text into sentences"""
    text = (text or "").strip()
    sentences = _SENT_SPLIT.split(text) if text else []
    return [s.strip() for s in sentences if s.strip()]


# ============================================================================
# SEMANTIC SIMILARITY MAPPING
# ============================================================================

class ClaimMapper:
    """
    Mappa claims di RAGAS (riformulati) alle frasi originali usando embeddings.
    """

    def __init__(self, model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
        """
        Args:
            model_name: Modello sentence-transformers (usa uno multilingue per italiano)
        """
        print(f"🔄 Caricamento modello {model_name}...")
        self.model = SentenceTransformer(model_name)
        print("✅ Modello caricato!")

    def find_best_match(self, claim: str, sentences: List[str], threshold: float = 0.3) -> Tuple[int, float, str]:
        """
        Trova la frase più simile al claim usando cosine similarity.

        Args:
            claim: Il claim estratto da RAGAS
            sentences: Lista di frasi del testo originale
            threshold: Soglia minima di similarità (0-1)

        Returns:
            (sentence_index, similarity_score, matched_sentence)
            Se nessun match: (-1, 0.0, "")
        """
        if not claim or not sentences:
            return (-1, 0.0, "")

        # Genera embeddings
        claim_emb = self.model.encode([claim])
        sent_embs = self.model.encode(sentences)

        # Calcola similarità
        similarities = cosine_similarity(claim_emb, sent_embs)[0]

        # Trova best match
        best_idx = np.argmax(similarities)
        best_score = similarities[best_idx]

        if best_score < threshold:
            return (-1, 0.0, "")

        return (int(best_idx), float(best_score), sentences[best_idx])

    def map_claim_to_position(self, claim: str, text: str) -> Dict[str, Any]:
        """
        Mappa un claim alla sua posizione nel testo.

        Returns:
            {
                'sentence_index': int,
                'matched_sentence': str,
                'similarity_score': float,
                'char_start': int,
                'char_end': int,
                'char_percentage': float,
                'position': str ('beginning', 'middle', 'end'),
                'total_sentences': int
            }
        """
        sentences = split_sentences(text)

        if not sentences:
            return {
                'sentence_index': -1,
                'matched_sentence': '',
                'similarity_score': 0.0,
                'char_start': -1,
                'char_end': -1,
                'char_percentage': -1.0,
                'position': 'unknown',
                'total_sentences': 0
            }

        # Trova match semantico
        sent_idx, score, matched_sent = self.find_best_match(claim, sentences)

        if sent_idx == -1:
            return {
                'sentence_index': -1,
                'matched_sentence': '',
                'similarity_score': 0.0,
                'char_start': -1,
                'char_end': -1,
                'char_percentage': -1.0,
                'position': 'unknown',
                'total_sentences': len(sentences)
            }

        # Trova posizione caratteri della frase matchata nel testo originale
        char_start = text.find(matched_sent)
        char_end = char_start + len(matched_sent) if char_start >= 0 else -1

        # Calcola percentuale
        char_percentage = (char_start / len(text) * 100) if len(text) > 0 and char_start >= 0 else -1.0

        # Categorizza posizione
        total_sents = len(sentences)
        if sent_idx < total_sents * 0.33:
            position = 'beginning'
        elif sent_idx < total_sents * 0.66:
            position = 'middle'
        else:
            position = 'end'

        return {
            'sentence_index': sent_idx,
            'matched_sentence': matched_sent,
            'similarity_score': score,
            'char_start': char_start,
            'char_end': char_end,
            'char_percentage': char_percentage,
            'position': position,
            'total_sentences': total_sents
        }

    def calculate_text_similarity(self, text1: str, text2: str) -> float:
        """
        Calcola similarità semantica tra due testi completi.

        Args:
            text1: Primo testo (es. statement FP)
            text2: Secondo testo (es. ground truth o contesto)

        Returns:
            Similarità da 0.0 a 1.0
        """
        if not text1 or not text2:
            return 0.0

        # Genera embeddings per i testi completi
        emb1 = self.model.encode([text1])
        emb2 = self.model.encode([text2])

        # Calcola cosine similarity
        similarity = cosine_similarity(emb1, emb2)[0][0]

        return float(similarity)

    def find_best_context_match(self, claim: str, contexts: List[str]) -> Dict[str, Any]:
        """
        Trova il contesto più simile al claim tra tutti i contesti disponibili.

        Args:
            claim: Il claim da cercare
            contexts: Lista di tutti i contesti

        Returns:
            {
                'best_context_index': int,
                'best_context_similarity': float,
                'best_context_text': str,
                'avg_contexts_similarity': float,
                'all_similarities': List[float]
            }
        """
        if not claim or not contexts:
            return {
                'best_context_index': -1,
                'best_context_similarity': 0.0,
                'best_context_text': '',
                'avg_contexts_similarity': 0.0,
                'all_similarities': []
            }

        # Calcola similarità con tutti i contesti
        claim_emb = self.model.encode([claim])
        contexts_embs = self.model.encode(contexts)

        similarities = cosine_similarity(claim_emb, contexts_embs)[0]

        # Trova il migliore
        best_idx = int(np.argmax(similarities))
        best_sim = float(similarities[best_idx])
        avg_sim = float(np.mean(similarities))

        return {
            'best_context_index': best_idx,
            'best_context_similarity': best_sim,
            'best_context_text': contexts[best_idx][:200] + '...' if len(contexts[best_idx]) > 200 else contexts[best_idx],
            'avg_contexts_similarity': avg_sim,
            'all_similarities': [float(s) for s in similarities]
        }


# ============================================================================
# PATTERN DETECTION
# ============================================================================

MONTHS_IT = r"(gennaio|febbraio|marzo|aprile|maggio|giugno|luglio|agosto|settembre|ottobre|novembre|dicembre)"
DATE_PATTERNS = [
    re.compile(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b"),
    re.compile(r"\b\d{4}\b"),
    re.compile(rf"\b\d{{1,2}}\s+{MONTHS_IT}\s+\d{{4}}\b", re.IGNORECASE),
]
NUMBER_PATTERN = re.compile(r"\b\d+(?:[.,]\d+)?\b")
PERCENT_PATTERN = re.compile(r"\b\d+(?:[.,]\d+)?\s*%\b")

def contains_date(text: str) -> bool:
    return any(p.search(text or "") for p in DATE_PATTERNS)

def contains_number(text: str) -> bool:
    return bool(NUMBER_PATTERN.search(text or ""))

def contains_percent(text: str) -> bool:
    return bool(PERCENT_PATTERN.search(text or ""))


# ============================================================================
# ERROR CATEGORIZATION
# ============================================================================

def categorize_fp_error(statement: str, reason: str) -> str:
    """Categorizza il tipo di False Positive"""
    statement_lower = statement.lower()
    reason_lower = str(reason).lower()

    if contains_date(statement):
        return 'date_error'

    names = ['gelli', 'musumeci', 'pazienza', 'fioravanti', 'mambro', 'picciafuoco', 'rinani']
    if any(name in statement_lower for name in names):
        return 'name_error'

    if any(word in statement_lower for word in ['condannato', 'assolto', 'sentenza', 'processo']):
        return 'legal_outcome_error'

    if any(k in reason_lower for k in ['non menzion', 'non presente', 'non support']):
        return 'fabricated_fact'

    if contains_number(statement) or contains_percent(statement):
        return 'numerical_error'

    return 'other'


def classify_hallucination(reason: Any) -> str:
    """Classifica inventata vs distorta"""
    if isinstance(reason, list):
        text = " ".join(str(r) for r in reason)
    else:
        text = str(reason)

    t = text.lower()

    if any(k in t for k in ["non menzion", "non presente", "assen", "nessuna evidenza", "non support"]):
        if any(k in t for k in ["dettaglio", "specific", "numero", "data", "non conferma", "parzial"]):
            return "distorta"
        return "inventata"

    return "inventata"


def estimate_fn_importance(statement: str, ground_truth: str, position_info: Dict) -> str:
    """Stima importanza di un False Negative"""
    important_keywords = [
        'condannato', 'assolto', 'sentenza', 'strage', 'gelli', 'articolo',
        'cassazione', 'appello', 'corte', 'processo'
    ]

    has_important = any(kw in statement.lower() for kw in important_keywords)
    is_at_beginning = (position_info.get('position') == 'beginning')
    is_high_similarity = (position_info.get('similarity_score', 0) > 0.7)

    if has_important and is_at_beginning:
        return 'high'
    elif has_important or is_at_beginning or is_high_similarity:
        return 'medium'
    else:
        return 'low'


# ============================================================================
# FP/FN ANALYSIS
# ============================================================================

def analyze_false_positives(result: Dict[str, Any],
                           response_text: str,
                           ground_truth: str,
                           top_k_contexts: List[str],
                           top_1_context: str,
                           mapper: ClaimMapper) -> List[Dict[str, Any]]:
    """
    Analizza False Positives con similarità semantica a GT e contesti.

    Args:
        result: Risultato RAGAS
        response_text: Risposta RAG completa
        ground_truth: Ground truth completa
        top_k_contexts: Lista di tutti i contesti recuperati
        top_1_context: Il miglior contesto (Top-1)
        mapper: ClaimMapper per calcolo similarità
    """
    fp_analysis = []

    pr_verdicts = result.get('pr_verdicts', {})
    response_vs_ref = pr_verdicts.get('response_vs_reference', [])

    for v in response_vs_ref:
        if v.get('verdict') == 0:  # False Positive
            statement = v.get('statement', '')
            reason = v.get('reason', '')

            # 1) Posizione nella risposta RAG (dove appare l'allucinazione)
            pos_info = mapper.map_claim_to_position(statement, response_text)

            # 2) NUOVO: Similarità semantica tra FP statement e ground truth
            similarity_to_gt = mapper.calculate_text_similarity(statement, ground_truth)

            # 3) NUOVO: Similarità tra FP statement e Top-1 context
            similarity_to_top1 = mapper.calculate_text_similarity(statement, top_1_context)

            # 4) NUOVO: Trova miglior match tra tutti i Top-k contexts
            context_analysis = mapper.find_best_context_match(statement, top_k_contexts)

            # Categorizza
            category = categorize_fp_error(statement, reason)
            tipo = classify_hallucination(reason)

            fp_analysis.append({
                'statement': statement,
                'reason': reason,
                'category': category,
                'tipo': tipo,

                # Posizione nella risposta
                'position': pos_info['position'],
                'sentence_index': pos_info['sentence_index'],
                'matched_sentence': pos_info['matched_sentence'],
                'similarity_score': pos_info['similarity_score'],
                'char_start': pos_info['char_start'],
                'char_end': pos_info['char_end'],
                'char_percentage': pos_info['char_percentage'],
                'total_sentences': pos_info['total_sentences'],

                # NUOVO: Similarità semantiche
                'similarity_fp_to_groundtruth': similarity_to_gt,
                'similarity_fp_to_top1_context': similarity_to_top1,
                'similarity_fp_to_best_topk_context': context_analysis['best_context_similarity'],
                'best_topk_context_index': context_analysis['best_context_index'],
                'avg_similarity_to_all_contexts': context_analysis['avg_contexts_similarity'],
                'best_topk_context_preview': context_analysis['best_context_text'],

                # Pattern detection
                'has_date': contains_date(statement),
                'has_number': contains_number(statement),
                'has_percent': contains_percent(statement)
            })

    return fp_analysis


def analyze_false_negatives(result: Dict[str, Any],
                           ground_truth: str,
                           mapper: ClaimMapper) -> List[Dict[str, Any]]:
    """
    Analizza False Negatives usando semantic similarity.
    """
    fn_analysis = []

    pr_verdicts = result.get('pr_verdicts', {})
    ref_vs_response = pr_verdicts.get('reference_vs_response', [])

    for v in ref_vs_response:
        if v.get('verdict') == 0:  # False Negative
            statement = v.get('statement', '')
            reason = v.get('reason', '')

            # Mappa claim alla posizione nella ground truth
            pos_info = mapper.map_claim_to_position(statement, ground_truth)

            # Stima importanza
            importance = estimate_fn_importance(statement, ground_truth, pos_info)

            fn_analysis.append({
                'statement': statement,
                'reason': reason,
                'importance': importance,
                'position': pos_info['position'],
                'sentence_index': pos_info['sentence_index'],
                'matched_sentence': pos_info['matched_sentence'],
                'similarity_score': pos_info['similarity_score'],
                'char_start': pos_info['char_start'],
                'char_end': pos_info['char_end'],
                'char_percentage': pos_info['char_percentage'],
                'total_sentences': pos_info['total_sentences'],
                'has_date': contains_date(statement),
                'has_number': contains_number(statement),
                'has_percent': contains_percent(statement)
            })

    return fn_analysis


# ============================================================================
# MAIN ANALYSIS
# ============================================================================

def analyze_all_results(results_json_path: str,
                       dataset_excel_path: str) -> Dict[str, Any]:
    """
    Analizza tutti i risultati RAGAS con similarità ai contesti.
    """
    # Carica dati
    print("📂 Caricamento dati...")
    with open(results_json_path, 'r', encoding='utf-8') as f:
        results = json.load(f)

    df = pd.read_excel(dataset_excel_path)

    # Inizializza mapper
    mapper = ClaimMapper()

    # Contatori
    all_fp = []
    all_fn = []

    fp_by_category = defaultdict(int)
    fp_by_position = defaultdict(int)
    fp_by_tipo = defaultdict(int)

    fn_by_importance = defaultdict(int)
    fn_by_position = defaultdict(int)

    print(f"\n🔍 Analisi di {len(results)} domande...\n")

    # Analizza ogni domanda
    for i, result in enumerate(results):
        question = result.get('Question', '')

        # Progress
        if (i + 1) % 5 == 0:
            print(f"   Processate {i + 1}/{len(results)} domande...")

        # Trova riga nel DataFrame
        row = df[df['Question'] == question]
        if row.empty:
            print(f"⚠️  Domanda {i+1} non trovata nel DataFrame")
            continue

        row = row.iloc[0]
        ground_truth = str(row.get('Ground_truth', ''))
        response = str(row.get('RAG_answer', ''))

        # NUOVO: Estrai contesti
        top_k_contexts_raw = str(row.get('Top_k_contexts_rag', ''))
        top_k_contexts = [ctx.strip() for ctx in top_k_contexts_raw.split('\n\n---\n\n') if ctx.strip()]

        top_1_context = str(row.get('Top_1_context_rag', ''))

        # Analizza FP con similarità ai contesti
        fp_list = analyze_false_positives(
            result, response, ground_truth,
            top_k_contexts, top_1_context, mapper
        )
        for fp in fp_list:
            fp['question_number'] = i + 1
            fp['question'] = question
            all_fp.append(fp)
            fp_by_category[fp['category']] += 1
            fp_by_position[fp['position']] += 1
            fp_by_tipo[fp['tipo']] += 1

        # Analizza FN (invariato)
        fn_list = analyze_false_negatives(result, ground_truth, mapper)
        for fn in fn_list:
            fn['question_number'] = i + 1
            fn['question'] = question
            all_fn.append(fn)
            fn_by_importance[fn['importance']] += 1
            fn_by_position[fn['position']] += 1

    print(f"\n✅ Analisi completata!\n")

    return {
        'total_fp': len(all_fp),
        'total_fn': len(all_fn),
        'fp_details': all_fp,
        'fn_details': all_fn,
        'fp_by_category': dict(fp_by_category),
        'fp_by_position': dict(fp_by_position),
        'fp_by_tipo': dict(fp_by_tipo),
        'fn_by_importance': dict(fn_by_importance),
        'fn_by_position': dict(fn_by_position)
    }


# ============================================================================
# REPORTING
# ============================================================================

def print_analysis_report(analysis: Dict[str, Any]):
    """Stampa report dettagliato"""

    print(f"\n{'='*80}")
    print("📊 ANALISI FP E FN (con Similarità ai Contesti)")
    print(f"{'='*80}\n")

    print(f"Totale False Positives: {analysis['total_fp']}")
    print(f"Totale False Negatives: {analysis['total_fn']}\n")

    # FALSE POSITIVES
    print(f"{'='*80}")
    print("❌ FALSE POSITIVES (Allucinazioni)")
    print(f"{'='*80}\n")

    print("📍 Distribuzione per Posizione nella Risposta:")
    for pos in ['beginning', 'middle', 'end', 'unknown']:
        count = analysis['fp_by_position'].get(pos, 0)
        pct = (count / analysis['total_fp'] * 100) if analysis['total_fp'] > 0 else 0
        print(f"  {pos:10s}: {count:3d} ({pct:5.1f}%)")

    print("\n🏷️  Distribuzione per Categoria:")
    for cat, count in sorted(analysis['fp_by_category'].items(), key=lambda x: x[1], reverse=True):
        pct = (count / analysis['total_fp'] * 100) if analysis['total_fp'] > 0 else 0
        print(f"  {cat:20s}: {count:3d} ({pct:5.1f}%)")

    # NUOVO: Statistiche similarità
    if analysis['fp_details']:
        avg_sim_gt = np.mean([fp['similarity_fp_to_groundtruth'] for fp in analysis['fp_details']])
        avg_sim_top1 = np.mean([fp['similarity_fp_to_top1_context'] for fp in analysis['fp_details']])
        avg_sim_best = np.mean([fp['similarity_fp_to_best_topk_context'] for fp in analysis['fp_details']])

        print("\n📊 Similarità Medie FP:")
        print(f"  FP → Ground Truth:     {avg_sim_gt:.3f}")
        print(f"  FP → Top-1 Context:    {avg_sim_top1:.3f}")
        print(f"  FP → Best Top-k:       {avg_sim_best:.3f}")

    # ESEMPI
    print(f"\n{'='*80}")
    print("🔍 ESEMPI DI FP CON SIMILARITÀ AI CONTESTI")
    print(f"{'='*80}\n")

    for i, fp in enumerate(analysis['fp_details'][:3], 1):
        print(f"{i}. Q{fp['question_number']} | {fp['category']} | Posizione: {fp['position']}")
        print(f"   Statement: {fp['statement'][:80]}...")
        print(f"   Similarità → GT:        {fp['similarity_fp_to_groundtruth']:.3f}")
        print(f"   Similarità → Top-1:     {fp['similarity_fp_to_top1_context']:.3f}")
        print(f"   Similarità → Best Top-k: {fp['similarity_fp_to_best_topk_context']:.3f}")
        print(f"   Best context preview: {fp['best_topk_context_preview'][:100]}...")
        print()


def export_to_excel(analysis: Dict[str, Any], output_path: str):
    """Esporta analisi in Excel"""

    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:

        # Summary
        summary_data = [
            {'Metric': 'Total FP', 'Count': analysis['total_fp']},
            {'Metric': 'Total FN', 'Count': analysis['total_fn']}
        ]

        # Aggiungi statistiche similarità
        if analysis['fp_details']:
            avg_sim_gt = np.mean([fp['similarity_fp_to_groundtruth'] for fp in analysis['fp_details']])
            avg_sim_top1 = np.mean([fp['similarity_fp_to_top1_context'] for fp in analysis['fp_details']])
            avg_sim_best = np.mean([fp['similarity_fp_to_best_topk_context'] for fp in analysis['fp_details']])

            summary_data.extend([
                {'Metric': 'Avg FP→GT Similarity', 'Count': f"{avg_sim_gt:.3f}"},
                {'Metric': 'Avg FP→Top1 Similarity', 'Count': f"{avg_sim_top1:.3f}"},
                {'Metric': 'Avg FP→BestTopK Similarity', 'Count': f"{avg_sim_best:.3f}"}
            ])

        summary = pd.DataFrame(summary_data)
        summary.to_excel(writer, sheet_name='Summary', index=False)

        # FP Details
        if analysis['fp_details']:
            df_fp = pd.DataFrame(analysis['fp_details'])
            df_fp = df_fp.sort_values('similarity_fp_to_groundtruth', ascending=False)
            df_fp.to_excel(writer, sheet_name='FP_Details', index=False)

        # FN Details
        if analysis['fn_details']:
            df_fn = pd.DataFrame(analysis['fn_details'])
            df_fn = df_fn.sort_values('similarity_score', ascending=False)
            df_fn.to_excel(writer, sheet_name='FN_Details', index=False)

        # Statistics
        stats = []
        for cat, count in analysis['fp_by_position'].items():
            stats.append({'Type': 'FP_Position', 'Category': cat, 'Count': count})
        for cat, count in analysis['fp_by_category'].items():
            stats.append({'Type': 'FP_Category', 'Category': cat, 'Count': count})
        for cat, count in analysis['fn_by_position'].items():
            stats.append({'Type': 'FN_Position', 'Category': cat, 'Count': count})
        for cat, count in analysis['fn_by_importance'].items():
            stats.append({'Type': 'FN_Importance', 'Category': cat, 'Count': count})

        df_stats = pd.DataFrame(stats)
        df_stats.to_excel(writer, sheet_name='Statistics', index=False)

    print(f"\n✅ Analisi esportata in: {output_path}")


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def run_complete_analysis(results_json_path: str,
                         dataset_excel_path: str,
                         output_excel_path: str = "fp_fn_analysis_with_context_similarity.xlsx"):
    """
    Esegue l'analisi completa con similarità ai contesti.
    """

    analysis = analyze_all_results(results_json_path, dataset_excel_path)
    print_analysis_report(analysis)
    export_to_excel(analysis, output_excel_path)

    return analysis


# ============================================================================
# ESEMPIO UTILIZZO
# ============================================================================

if __name__ == "__main__":
    analysis = run_complete_analysis(
        results_json_path="/content/drive/MyDrive/ESAMI/AI/ragas_results_70_ok_solo.json",
        dataset_excel_path="/content/drive/MyDrive/ESAMI/AI/dataset_with_source_aware_rag.xlsx",
        output_excel_path="/content/drive/MyDrive/ESAMI/AI/fp_fn_with_context_similarity.xlsx"
    )

    print(f"\n📊 RISULTATI FINALI:")
    print(f"Total FP: {analysis['total_fp']}")
    print(f"Total FN: {analysis['total_fn']}")

📂 Caricamento dati...
🔄 Caricamento modello sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Modello caricato!

🔍 Analisi di 20 domande...

   Processate 5/20 domande...
   Processate 10/20 domande...
   Processate 15/20 domande...
   Processate 20/20 domande...

✅ Analisi completata!


📊 ANALISI FP E FN (con Similarità ai Contesti)

Totale False Positives: 26
Totale False Negatives: 45

❌ FALSE POSITIVES (Allucinazioni)

📍 Distribuzione per Posizione nella Risposta:
  beginning :  17 ( 65.4%)
  middle    :   6 ( 23.1%)
  end       :   3 ( 11.5%)
  unknown   :   0 (  0.0%)

🏷️  Distribuzione per Categoria:
  other               :  14 ( 53.8%)
  name_error          :   7 ( 26.9%)
  legal_outcome_error :   4 ( 15.4%)
  date_error          :   1 (  3.8%)

📊 Similarità Medie FP:
  FP → Ground Truth:     0.360
  FP → Top-1 Context:    0.394
  FP → Best Top-k:       0.430

🔍 ESEMPI DI FP CON SIMILARITÀ AI CONTESTI

1. Q1 | name_error | Posizione: beginning
   Statement: Picciafuoco si avvicinò a gruppi di estrema destra....
   Similarità → GT:        0.268
   Similarità → Top-1:  